# Resonance: fine-tune, evaluate, publish

Runs the GPU part of [resonance-physics](https://github.com/victoriaT19/resonance-physics) on a free Colab T4
(Runtime → Change runtime type → T4 GPU). All logic lives in the `resonance` package; this notebook only calls it.

The training set (`data/generated/`) is built beforehand without a GPU (`python -m resonance.data build`) and
committed to the repository, so this notebook does not need the teacher.

1. QLoRA fine-tuning of Gemma 4 E2B with Unsloth (`resonance.train`), about 1-2 h on a T4
2. Base vs fine-tuned evaluation on 20 held-out questions (`resonance.evaluate`), about 30-40 min
3. Model card from the measured numbers, upload to the Hub (`resonance.card`)

**Colab secrets** (key icon on the left):

| name | needed for |
|---|---|
| `HF_TOKEN` | step 3: a Hugging Face token with write access |
| `GEMINI_API_KEY` | optional, step 2: the LLM judge (Gemma 4 31B through the Gemini API) |

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!git clone -q -b fix/treino-avaliacao-demo https://github.com/victoriaT19/resonance-physics.git
%cd resonance-physics
!pip install -q unsloth
!pip install -q -r requirements-train.txt

In [ ]:
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

for name in ["HF_TOKEN", "GEMINI_API_KEY"]:
    value = secret(name)
    if value:
        os.environ[name] = value
print("secrets set:", [n for n in ["HF_TOKEN", "GEMINI_API_KEY"] if os.environ.get(n)])

In [ ]:
import json
print(json.dumps(json.load(open("data/generated/stats.json", encoding="utf-8")), indent=2, ensure_ascii=False))
!python -m pytest -q

## 1. Fine-tuning

In [ ]:
!python -m resonance.train --data data/generated --out outputs/adapter

## 2. Evaluation: base vs fine-tuned

In [ ]:
judge = "" if os.environ.get("GEMINI_API_KEY") else "--no-judge"
!python -m resonance.evaluate --adapter outputs/adapter --out results/v2 --load-in-4bit {judge}
from IPython.display import Markdown
Markdown(open("results/v2/table.md", encoding="utf-8").read())

## 3. Save and publish

Zips the evaluation and training log for the repository, writes the model card from the measured numbers and
uploads the adapter. Skip the last line to look at the card before publishing.

In [ ]:
!cp outputs/adapter/training_log.json results/v2/
!zip -qr results_v2.zip results/v2
try:
    from google.colab import files
    files.download("results_v2.zip")
except Exception:
    print("results_v2.zip is in the working directory")

In [ ]:
!python -m resonance.card --adapter outputs/adapter --data data/generated --eval results/v2 --verified --repo vtaua/resonance-gemma4-physics
print(open("outputs/adapter/README.md", encoding="utf-8").read()[:3000])

In [ ]:
!python -m resonance.card --adapter outputs/adapter --data data/generated --eval results/v2 --verified --repo vtaua/resonance-gemma4-physics --push

## 4. Try the demo (optional)

In [ ]:
!python app.py --adapter outputs/adapter --load-in-4bit --share